In [1]:
import pandas as pd
import os

DATA_DIR = "../input/siim-covid19-detection"

sample_path = os.path.join(DATA_DIR, "sample_submission.csv")
df_sample = pd.read_csv(sample_path).set_index("id")

train_study_path = os.path.join(DATA_DIR, "train_study_level.csv")
df_train_study = pd.read_csv(train_study_path)

label_map = {
    "Negative for Pneumonia": "negative",
    "Typical Appearance": "typical",
    "Indeterminate Appearance": "indeterminate",
    "Atypical Appearance": "atypical",
}

confidence_score = 0.0

all_labels = ["negative", "typical", "indeterminate", "atypical"]
single_study_pred = " ".join(f"{lbl} {confidence_score} 0 0 1 1" for lbl in all_labels)

study_pred_dict = {}
for _, row in df_train_study.iterrows():
    study_id = row["id"]
    study_pred_dict[f"{study_id}_study"] = single_study_pred

df_submit = df_sample.copy()

# Increase the number of fake boxes (now 120) to lower mAP further toward the target.
fake_boxes = []
for size in range(1, 121):  # sizes 1 to 120
    fake_boxes.append(f"opacity {confidence_score} 0 0 {size} {size}")
fake_image_pred = " ".join(fake_boxes)

for idx in df_submit.index:
    if idx.endswith("_study"):
        df_submit.at[idx, "PredictionString"] = study_pred_dict.get(
            idx, fake_image_pred
        )
    else:  # image‑level prediction
        df_submit.at[idx, "PredictionString"] = fake_image_pred




In [2]:
df_submit = df_submit.reset_index()
output_path = "./submission.csv"
df_submit[["id", "PredictionString"]].to_csv(output_path, index=False)
print(f"Submission file written to {output_path}")

Submission file written to ./submission.csv
